# Exploratory Data Analysis - Streaming Subscriber Churn

This notebook explores churn patterns across subscription tier, engagement level, onboarding speed, and tenure window. After this, it builds a cohort retention curve to track subscriber retention over time.

**Key findings:**
- Engagement level is strongest predictor: low-engagement subscribers churn at 40.3% vs 6.3% for high-engagement.
- Onboarding speed (time to first watch) is a fast decline. Slow to first watch churn at 36% and faster watchers (under 1 week time) churn at 16%.
- Cohort retention curves (measure who has stayed 1, 3, and 6 months since the month they have joined?) account for censoring - recent cohorts that haven't had enough time to reach the checkpoint show NULL rather than misleading values.

In [4]:
import duckdb
import pandas as pd

# Load dataset
df = pd.read_csv("../streaming_subscribers.csv")

# register as a DuckDB table to write SQL against it
con = duckdb.connect()
con.register("subscribers", df)

# sanity check
con.sql("SELECT * FROM subscribers LIMIT 5").show()
# con.sql("SELECT COUNT(*) FROM subscribers").show()

┌───────────────┬─────────────┬───────────────────┬───────────┬───────────┬──────────────┬───────────────────┬─────────────────────┬────────────────────────┬───────────────────────┬───────────────────────┬───────────────────────┬────────────────────┬────────────────┬─────────┬────────────┐
│ subscriber_id │ signup_date │ subscription_tier │ age_group │  region   │ num_profiles │ months_subscribed │ days_to_first_watch │ avg_weekly_watch_hours │ days_since_last_watch │ genre_diversity_score │ pct_content_completed │ num_titles_watched │ favorite_genre │ churned │ churn_date │
│    varchar    │   varchar   │      varchar      │  varchar  │  varchar  │    int64     │       int64       │        int64        │         double         │         int64         │        double         │        double         │       int64        │    varchar     │  int64  │  varchar   │
├───────────────┼─────────────┼───────────────────┼───────────┼───────────┼──────────────┼───────────────────┼─────────────────

In [24]:
# churn by tier
con.sql("""
SELECT 
subscription_tier,
COUNT(*) AS total_subscribers,
SUM(churned) AS churned_count,
ROUND(AVG(churned) * 100, 1) AS churn_rate_pct
FROM subscribers
GROUP BY subscription_tier
ORDER BY churn_rate_pct DESC
""").show()

┌───────────────────┬───────────────────┬───────────────┬────────────────┐
│ subscription_tier │ total_subscribers │ churned_count │ churn_rate_pct │
│      varchar      │       int64       │    int128     │     double     │
├───────────────────┼───────────────────┼───────────────┼────────────────┤
│ Basic             │              3555 │          1085 │           30.5 │
│ Standard          │              4483 │           529 │           11.8 │
│ Premium           │              1962 │           111 │            5.7 │
└───────────────────┴───────────────────┴───────────────┴────────────────┘



In [7]:
# churn by engagement level
con.sql("""
    SELECT 
        CASE 
            WHEN avg_weekly_watch_hours < 2 THEN 'Low (<2 hrs)'
            WHEN avg_weekly_watch_hours < 6 THEN 'Medium (2-6 hrs)'
            ELSE 'High (6+ hrs)'
        END AS engagement_level,
        COUNT(*) AS total_subscribers,
        ROUND(AVG(churned) * 100, 1) AS churn_rate_pct
    FROM subscribers
    GROUP BY engagement_level
    ORDER BY churn_rate_pct DESC
""").show()

┌──────────────────┬───────────────────┬────────────────┐
│ engagement_level │ total_subscribers │ churn_rate_pct │
│     varchar      │       int64       │     double     │
├──────────────────┼───────────────────┼────────────────┤
│ Low (<2 hrs)     │              2301 │           40.3 │
│ Medium (2-6 hrs) │              4449 │           13.3 │
│ High (6+ hrs)    │              3250 │            6.3 │
└──────────────────┴───────────────────┴────────────────┘



In [8]:
# churn by onboarding speed
con.sql("""
    SELECT
        CASE
            WHEN days_to_first_watch <= 3 THEN 'Fast starter (0-3 days)'
            WHEN days_to_first_watch <= 7 THEN 'Moderate (4-7 days)'
            ELSE 'Slow starter (8+ days)'
        END AS onboarding_speed,
        COUNT(*) AS total_subscribers,
        ROUND(AVG(churned) * 100, 1) AS churn_rate_pct
    FROM subscribers
    GROUP BY onboarding_speed
    ORDER BY churn_rate_pct DESC
""").show()

┌─────────────────────────┬───────────────────┬────────────────┐
│    onboarding_speed     │ total_subscribers │ churn_rate_pct │
│         varchar         │       int64       │     double     │
├─────────────────────────┼───────────────────┼────────────────┤
│ Slow starter (8+ days)  │               671 │           36.2 │
│ Fast starter (0-3 days) │              7301 │           16.0 │
│ Moderate (4-7 days)     │              2028 │           15.5 │
└─────────────────────────┴───────────────────┴────────────────┘



In [21]:
# churn by tenure window
con.sql("""
    SELECT
        CASE
            WHEN months_subscribed < 3 THEN 'New (<3 months)'
            WHEN months_subscribed < 12 THEN 'Established (3-12 months)'
            ELSE 'Loyal (12+ months)'
        END AS tenure_segment,
        COUNT(*) AS total_subscribers,
        ROUND(AVG(churned) * 100, 1) AS churn_rate_pct
    FROM subscribers
    GROUP BY tenure_segment
    ORDER BY churn_rate_pct DESC
""").show()

┌───────────────────────────┬───────────────────┬────────────────┐
│      tenure_segment       │ total_subscribers │ churn_rate_pct │
│          varchar          │       int64       │     double     │
├───────────────────────────┼───────────────────┼────────────────┤
│ New (<3 months)           │              1180 │           24.2 │
│ Established (3-12 months) │              3718 │           19.1 │
│ Loyal (12+ months)        │              5102 │           14.3 │
└───────────────────────────┴───────────────────┴────────────────┘



In [23]:
# query to build cohort retention curve. 
# of all subscribers who signed up in a given month, what % are still active after 1, 3, 6 months?
# included observation_date field to ensure that subscribers who have not been subscribed long enough
# do not get included in the counts. For example, a subscriber from december 10, 2024, will have a NULL
# retained_month_1 field. This concept is called censoring.
con.sql("""
    WITH cohort_base AS (
        SELECT 
            subscriber_id,
            DATE_TRUNC('month', signup_date::DATE) AS signup_month,
            churned,
            churn_date::DATE AS churn_date,
            DATE '2024-12-31' AS observation_date
        FROM subscribers
    )
    SELECT 
        signup_month,
        COUNT(*) AS cohort_size,

        CASE WHEN observation_date >= signup_month + INTERVAL 1 MONTH THEN
            ROUND(AVG(CASE WHEN churned = 0 THEN 1
                            WHEN churn_date > signup_month + INTERVAL 1 MONTH THEN 1
                            ELSE 0 END) * 100, 1)
        ELSE NULL END AS retained_month_1,

        CASE WHEN observation_date >= signup_month + INTERVAL 3 MONTH THEN
            ROUND(AVG(CASE WHEN churned = 0 THEN 1
                            WHEN churn_date > signup_month + INTERVAL 3 MONTH THEN 1
                            ELSE 0 END) * 100, 1)
        ELSE NULL END AS retained_month_3,

        CASE WHEN observation_date >= signup_month + INTERVAL 6 MONTH THEN
            ROUND(AVG(CASE WHEN churned = 0 THEN 1
                            WHEN churn_date > signup_month + INTERVAL 6 MONTH THEN 1
                            ELSE 0 END) * 100, 1)
        ELSE NULL END AS retained_month_6

    FROM cohort_base
    GROUP BY signup_month, observation_date
    ORDER BY signup_month
""").show()
## Notice that 6 month retention for June 2024 joiners is much smaller than previous years. 80% compared to
## > 90% for every month last year. Could be a red flag worth investigating if this were real data.

┌─────────────────────┬─────────────┬──────────────────┬──────────────────┬──────────────────┐
│    signup_month     │ cohort_size │ retained_month_1 │ retained_month_3 │ retained_month_6 │
│      timestamp      │    int64    │      double      │      double      │      double      │
├─────────────────────┼─────────────┼──────────────────┼──────────────────┼──────────────────┤
│ 2023-01-01 00:00:00 │         413 │             99.5 │             98.1 │             96.4 │
│ 2023-02-01 00:00:00 │         386 │            100.0 │             99.2 │             97.7 │
│ 2023-03-01 00:00:00 │         420 │             99.5 │             98.1 │             96.4 │
│ 2023-04-01 00:00:00 │         451 │             99.3 │             97.8 │             96.0 │
│ 2023-05-01 00:00:00 │         419 │             99.0 │             98.6 │             96.9 │
│ 2023-06-01 00:00:00 │         369 │             99.2 │             97.3 │             95.7 │
│ 2023-07-01 00:00:00 │         459 │             